# 1. Loading dataset

In [1]:
from pyspark.sql import SparkSession, Row
from pyspark.sql.functions import col, lit, when, count, avg
from pyspark.ml.feature import StringIndexer, OneHotEncoder
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import StructType, StructField, StringType

### Load dataset

In [2]:
spark = SparkSession.builder.appName("m3_spark").getOrCreate()

df = spark.read.parquet("/data/fintech_data_18_52_11870.parquet")

### Preview first 20 rows

In [3]:
df.show()

+--------------------+--------------------+----------+--------------+----------+----------------+-------------------+--------+----------+-----------+-----------+-------+-----------+-----------+-----+-------------+----------+--------+-----+-----------------+----------+----------+------------------+--------------------+
|         Customer Id|           Emp Title|Emp Length|Home Ownership|Annual Inc|Annual Inc Joint|Verification Status|Zip Code|Addr State|Avg Cur Bal|Tot Cur Bal|Loan Id|Loan Status|Loan Amount|State|Funded Amount|      Term|Int Rate|Grade|       Issue Date|Pymnt Plan|      Type|           Purpose|         Description|
+--------------------+--------------------+----------+--------------+----------+----------------+-------------------+--------+----------+-----------+-----------+-------+-----------+-----------+-----+-------------+----------+--------+-----+-----------------+----------+----------+------------------+--------------------+
|Yid6XHg5Y1x4YWRce...|       Store Manag

### How many partitions is this dataframe split into

In [4]:
df.rdd.getNumPartitions()

1

### Change partitions to be equal to number of logical cores

In [5]:
!pip install psutil
import psutil

logical_cores = psutil.cpu_count(logical=True)
print(f"Logical cores: {logical_cores}")

Logical cores: 12


In [6]:
df = df.repartition(logical_cores)
df.rdd.getNumPartitions()

12

# 2. Cleaning

### Rename Columns

In [7]:
df = df.toDF(*[col.replace(" ", "_").lower() for col in df.columns])
df.columns

['customer_id',
 'emp_title',
 'emp_length',
 'home_ownership',
 'annual_inc',
 'annual_inc_joint',
 'verification_status',
 'zip_code',
 'addr_state',
 'avg_cur_bal',
 'tot_cur_bal',
 'loan_id',
 'loan_status',
 'loan_amount',
 'state',
 'funded_amount',
 'term',
 'int_rate',
 'grade',
 'issue_date',
 'pymnt_plan',
 'type',
 'purpose',
 'description']

### Detect Missing

In [8]:
def detect_missing(df):
    missing_info = {}
    num_rows = df.count()
    for col in df.columns:
        missing_info[col] = f"{round(df.filter(df[col].isNull()).count() / num_rows * 100, 2)}%" 

    data = list(missing_info.items())

    missing_df = spark.createDataFrame(data, schema=["col_name", "missing_percentage"])

    return missing_info, missing_df

In [9]:
missing_values_dict, missing_df = detect_missing(df)
missing_df.show()

+-------------------+------------------+
|           col_name|missing_percentage|
+-------------------+------------------+
|        customer_id|              0.0%|
|          emp_title|             8.58%|
|         emp_length|             6.69%|
|     home_ownership|              0.0%|
|         annual_inc|              0.0%|
|   annual_inc_joint|            92.99%|
|verification_status|              0.0%|
|           zip_code|              0.0%|
|         addr_state|              0.0%|
|        avg_cur_bal|              0.0%|
|        tot_cur_bal|              0.0%|
|            loan_id|              0.0%|
|        loan_status|              0.0%|
|        loan_amount|              0.0%|
|              state|              0.0%|
|      funded_amount|              0.0%|
|               term|              0.0%|
|           int_rate|             4.41%|
|              grade|              0.0%|
|         issue_date|              0.0%|
+-------------------+------------------+
only showing top

### Handle Missing Values

In [10]:
for col, dtype in df.dtypes:
    if dtype == 'double' or dtype == 'bigint':
        df = df.fillna({col: 0})
    else:
        col_mode_df = df.groupBy(col).count().orderBy("count", ascending=False)
        first_row = col_mode_df.first()
        mode_val = first_row[0]
        if first_row[0] is None:
            mode_val = col_mode_df.collect()[1][0]      
        df = df.fillna({col: mode_val})

### Check Missing After Imputation

In [11]:
missing_values_dict_after, missing_df_after = detect_missing(df)
missing_df_after.show()

+-------------------+------------------+
|           col_name|missing_percentage|
+-------------------+------------------+
|        customer_id|              0.0%|
|          emp_title|              0.0%|
|         emp_length|              0.0%|
|     home_ownership|              0.0%|
|         annual_inc|              0.0%|
|   annual_inc_joint|              0.0%|
|verification_status|              0.0%|
|           zip_code|              0.0%|
|         addr_state|              0.0%|
|        avg_cur_bal|              0.0%|
|        tot_cur_bal|              0.0%|
|            loan_id|              0.0%|
|        loan_status|              0.0%|
|        loan_amount|              0.0%|
|              state|              0.0%|
|      funded_amount|              0.0%|
|               term|              0.0%|
|           int_rate|              0.0%|
|              grade|              0.0%|
|         issue_date|              0.0%|
+-------------------+------------------+
only showing top

## Creating lookup table to fill it while encoding

In [12]:
lookup_schema = StructType([StructField("col_name", StringType(), False), StructField("original_value", StringType(), True), StructField("encoded_value", StringType(), False)])

lookup_df = spark.createDataFrame([], schema=lookup_schema)

In [13]:
def add_to_lookup_table(lookup_df, feature_name, mapping):
    new_mappings = spark.createDataFrame(
        [Row(feature_name=feature_name, original_value=k, encoded_value=v) for k, v in mapping.items()]
    )
    updated_lookup_df = lookup_df.union(new_mappings)
    return updated_lookup_df

# 3. Encoding

### Emp Length

In [14]:
from pyspark.sql.functions import when, col

df = df.withColumn(
    "emp_length_encoded", 
    when(col("emp_length") == "< 1 year", 0)
    .when(col("emp_length") == "1 year", 1)
    .when(col("emp_length") == "2 years", 2)
    .when(col("emp_length") == "3 years", 3)
    .when(col("emp_length") == "4 years", 4)
    .when(col("emp_length") == "5 years", 5)
    .when(col("emp_length") == "6 years", 6)
    .when(col("emp_length") == "7 years", 7)
    .when(col("emp_length") == "8 years", 8)
    .when(col("emp_length") == "9 years", 9)
    .when(col("emp_length") == "10+ years", 10)
    .otherwise(None)
)

In [15]:
emp_length_mapping = {
    "< 1 year": 0,
    "1 year": 1,
    "2 years": 2,
    "3 years": 3,
    "4 years": 4,
    "5 years": 5,
    "6 years": 6,
    "7 years": 7,
    "8 years": 8,
    "9 years": 9,
    "10+ years": 10
}

lookup_df = add_to_lookup_table(lookup_df, 'emp_length', emp_length_mapping)
    

## One Hot Encoding

In [16]:
def one_hot_encoding_helper(df, col_name, value):
    return df.withColumn(
        f"{col_name}_{value}",
        when(col(col_name) == value, 1)
        .otherwise(0)
    )

In [17]:
def one_hot_encoding(df, col_name):
    unique_values = df.select(col_name).distinct().collect()

    for row in unique_values:
        df = one_hot_encoding_helper(df, col_name, row[col_name])
    return df

### Home Ownership

In [18]:
df = one_hot_encoding(df, 'home_ownership')
df.show()

+--------------------+--------------------+----------+--------------+----------+----------------+-------------------+--------+----------+-----------+-----------+-------+-----------+-----------+-----+-------------+----------+--------+-----+-----------------+----------+----------+------------------+--------------------+------------------+------------------+-------------------+-----------------------+------------------+--------------------+
|         customer_id|           emp_title|emp_length|home_ownership|annual_inc|annual_inc_joint|verification_status|zip_code|addr_state|avg_cur_bal|tot_cur_bal|loan_id|loan_status|loan_amount|state|funded_amount|      term|int_rate|grade|       issue_date|pymnt_plan|      type|           purpose|         description|emp_length_encoded|home_ownership_OWN|home_ownership_RENT|home_ownership_MORTGAGE|home_ownership_ANY|home_ownership_OTHER|
+--------------------+--------------------+----------+--------------+----------+----------------+-------------------

### Verification Status

In [19]:
df = one_hot_encoding(df, 'verification_status')
df.show()

+--------------------+--------------------+----------+--------------+----------+----------------+-------------------+--------+----------+-----------+-----------+-------+-----------+-----------+-----+-------------+----------+--------+-----+-----------------+----------+----------+------------------+--------------------+------------------+------------------+-------------------+-----------------------+------------------+--------------------+----------------------------+-----------------------------------+--------------------------------+
|         customer_id|           emp_title|emp_length|home_ownership|annual_inc|annual_inc_joint|verification_status|zip_code|addr_state|avg_cur_bal|tot_cur_bal|loan_id|loan_status|loan_amount|state|funded_amount|      term|int_rate|grade|       issue_date|pymnt_plan|      type|           purpose|         description|emp_length_encoded|home_ownership_OWN|home_ownership_RENT|home_ownership_MORTGAGE|home_ownership_ANY|home_ownership_OTHER|verification_status

### Type

In [20]:
df = one_hot_encoding(df, 'type')
df.show()

+--------------------+--------------------+----------+--------------+----------+----------------+-------------------+--------+----------+-----------+-----------+-------+-----------+-----------+-----+-------------+----------+--------+-----+-----------------+----------+----------+------------------+--------------------+------------------+------------------+-------------------+-----------------------+------------------+--------------------+----------------------------+-----------------------------------+--------------------------------+--------------+---------------+---------------+----------+
|         customer_id|           emp_title|emp_length|home_ownership|annual_inc|annual_inc_joint|verification_status|zip_code|addr_state|avg_cur_bal|tot_cur_bal|loan_id|loan_status|loan_amount|state|funded_amount|      term|int_rate|grade|       issue_date|pymnt_plan|      type|           purpose|         description|emp_length_encoded|home_ownership_OWN|home_ownership_RENT|home_ownership_MORTGAGE|h

### State

In [21]:
state_indexer = StringIndexer(inputCol="state", outputCol="state_index").fit(df)
df = state_indexer.transform(df)
df.show()

+--------------------+--------------------+----------+--------------+----------+----------------+-------------------+--------+----------+-----------+-----------+-------+-----------+-----------+-----+-------------+----------+--------+-----+-----------------+----------+----------+------------------+--------------------+------------------+------------------+-------------------+-----------------------+------------------+--------------------+----------------------------+-----------------------------------+--------------------------------+--------------+---------------+---------------+----------+-----------+
|         customer_id|           emp_title|emp_length|home_ownership|annual_inc|annual_inc_joint|verification_status|zip_code|addr_state|avg_cur_bal|tot_cur_bal|loan_id|loan_status|loan_amount|state|funded_amount|      term|int_rate|grade|       issue_date|pymnt_plan|      type|           purpose|         description|emp_length_encoded|home_ownership_OWN|home_ownership_RENT|home_ownershi

In [22]:
df = df.withColumn("state_index", col('state_index').cast('int'))
df.show()

+--------------------+--------------------+----------+--------------+----------+----------------+-------------------+--------+----------+-----------+-----------+-------+-----------+-----------+-----+-------------+----------+--------+-----+-----------------+----------+----------+------------------+--------------------+------------------+------------------+-------------------+-----------------------+------------------+--------------------+----------------------------+-----------------------------------+--------------------------------+--------------+---------------+---------------+----------+-----------+
|         customer_id|           emp_title|emp_length|home_ownership|annual_inc|annual_inc_joint|verification_status|zip_code|addr_state|avg_cur_bal|tot_cur_bal|loan_id|loan_status|loan_amount|state|funded_amount|      term|int_rate|grade|       issue_date|pymnt_plan|      type|           purpose|         description|emp_length_encoded|home_ownership_OWN|home_ownership_RENT|home_ownershi

In [23]:
mapping = dict(enumerate(state_indexer.labels))

state_mapping = {v: k for k, v in mapping.items()}

lookup_df = add_to_lookup_table(lookup_df, 'state', state_mapping)
lookup_df.show()

+----------+--------------+-------------+
|  col_name|original_value|encoded_value|
+----------+--------------+-------------+
|emp_length|      < 1 year|            0|
|emp_length|        1 year|            1|
|emp_length|       2 years|            2|
|emp_length|       3 years|            3|
|emp_length|       4 years|            4|
|emp_length|       5 years|            5|
|emp_length|       6 years|            6|
|emp_length|       7 years|            7|
|emp_length|       8 years|            8|
|emp_length|       9 years|            9|
|emp_length|     10+ years|           10|
|     state|            CA|            0|
|     state|            TX|            1|
|     state|            NY|            2|
|     state|            FL|            3|
|     state|            IL|            4|
|     state|            NJ|            5|
|     state|            PA|            6|
|     state|            OH|            7|
|     state|            GA|            8|
+----------+--------------+-------

### Purpose

In [24]:
purpose_indexer = StringIndexer(inputCol="purpose", outputCol="purpose_index").fit(df)
df = purpose_indexer.transform(df)
df.show()

+--------------------+--------------------+----------+--------------+----------+----------------+-------------------+--------+----------+-----------+-----------+-------+-----------+-----------+-----+-------------+----------+--------+-----+-----------------+----------+----------+------------------+--------------------+------------------+------------------+-------------------+-----------------------+------------------+--------------------+----------------------------+-----------------------------------+--------------------------------+--------------+---------------+---------------+----------+-----------+-------------+
|         customer_id|           emp_title|emp_length|home_ownership|annual_inc|annual_inc_joint|verification_status|zip_code|addr_state|avg_cur_bal|tot_cur_bal|loan_id|loan_status|loan_amount|state|funded_amount|      term|int_rate|grade|       issue_date|pymnt_plan|      type|           purpose|         description|emp_length_encoded|home_ownership_OWN|home_ownership_RENT

In [25]:
df = df.withColumn("purpose_index", col('purpose_index').cast('int'))
df.show()

+--------------------+--------------------+----------+--------------+----------+----------------+-------------------+--------+----------+-----------+-----------+-------+-----------+-----------+-----+-------------+----------+--------+-----+-----------------+----------+----------+------------------+--------------------+------------------+------------------+-------------------+-----------------------+------------------+--------------------+----------------------------+-----------------------------------+--------------------------------+--------------+---------------+---------------+----------+-----------+-------------+
|         customer_id|           emp_title|emp_length|home_ownership|annual_inc|annual_inc_joint|verification_status|zip_code|addr_state|avg_cur_bal|tot_cur_bal|loan_id|loan_status|loan_amount|state|funded_amount|      term|int_rate|grade|       issue_date|pymnt_plan|      type|           purpose|         description|emp_length_encoded|home_ownership_OWN|home_ownership_RENT

In [26]:
mapping = dict(enumerate(purpose_indexer.labels))

purpose_mapping = {v: k for k, v in mapping.items()}

lookup_df = add_to_lookup_table(lookup_df, 'purpose', purpose_mapping)
lookup_df.show()

+----------+--------------+-------------+
|  col_name|original_value|encoded_value|
+----------+--------------+-------------+
|emp_length|      < 1 year|            0|
|emp_length|        1 year|            1|
|emp_length|       2 years|            2|
|emp_length|       3 years|            3|
|emp_length|       4 years|            4|
|emp_length|       5 years|            5|
|emp_length|       6 years|            6|
|emp_length|       7 years|            7|
|emp_length|       8 years|            8|
|emp_length|       9 years|            9|
|emp_length|     10+ years|           10|
|     state|            CA|            0|
|     state|            TX|            1|
|     state|            NY|            2|
|     state|            FL|            3|
|     state|            IL|            4|
|     state|            NJ|            5|
|     state|            PA|            6|
|     state|            OH|            7|
|     state|            GA|            8|
+----------+--------------+-------

### Grade

In [27]:
df = df.withColumn(
    "letter_grade",
    when((col("grade") >= 1) & (col("grade") <= 5), "A")
    .when((col("grade") >= 6) & (col("grade") <= 10), "B")
    .when((col("grade") >= 11) & (col("grade") <= 15), "C")
    .when((col("grade") >= 16) & (col("grade") <= 20), "D")
    .when((col("grade") >= 21) & (col("grade") <= 25), "E")
    .when((col("grade") >= 26) & (col("grade") <= 30), "F")
    .when((col("grade") >= 31) & (col("grade") <= 35), "G")
    .otherwise(None)
)

In [28]:
df.show()

+--------------------+--------------------+----------+--------------+----------+----------------+-------------------+--------+----------+-----------+-----------+-------+-----------+-----------+-----+-------------+----------+--------+-----+-----------------+----------+----------+------------------+--------------------+------------------+------------------+-------------------+-----------------------+------------------+--------------------+----------------------------+-----------------------------------+--------------------------------+--------------+---------------+---------------+----------+-----------+-------------+------------+
|         customer_id|           emp_title|emp_length|home_ownership|annual_inc|annual_inc_joint|verification_status|zip_code|addr_state|avg_cur_bal|tot_cur_bal|loan_id|loan_status|loan_amount|state|funded_amount|      term|int_rate|grade|       issue_date|pymnt_plan|      type|           purpose|         description|emp_length_encoded|home_ownership_OWN|home_o

In [29]:
grade_letter_mapping = {
    (1, 5): "A",
    (6, 10): "B",
    (11, 15): "C",
    (16, 20): "D",
    (21, 25): "E",
    (26, 30): "F",
    (31, 35): "G"
}

expanded_mapping = {}
for grade_range, letter in grade_letter_mapping.items():
    for grade in range(grade_range[0], grade_range[1] + 1):
        expanded_mapping[str(grade)] = letter

lookup_df = add_to_lookup_table(lookup_df, "grade", expanded_mapping)
lookup_df.show()

+----------+--------------+-------------+
|  col_name|original_value|encoded_value|
+----------+--------------+-------------+
|emp_length|      < 1 year|            0|
|emp_length|        1 year|            1|
|emp_length|       2 years|            2|
|emp_length|       3 years|            3|
|emp_length|       4 years|            4|
|emp_length|       5 years|            5|
|emp_length|       6 years|            6|
|emp_length|       7 years|            7|
|emp_length|       8 years|            8|
|emp_length|       9 years|            9|
|emp_length|     10+ years|           10|
|     state|            CA|            0|
|     state|            TX|            1|
|     state|            NY|            2|
|     state|            FL|            3|
|     state|            IL|            4|
|     state|            NJ|            5|
|     state|            PA|            6|
|     state|            OH|            7|
|     state|            GA|            8|
+----------+--------------+-------

# 4. Feature Engineering

### Previous loan issue date from the same grade

In [30]:
df = df.withColumn("issue_date", F.to_date(F.to_timestamp(F.col("issue_date"), "d MMMM yyyy")))
window = Window.partitionBy('grade').orderBy('issue_date')

In [31]:
df = df.withColumn('prev_issue_date_per_grade', F.lag('issue_date').over(window))
df = df.withColumn("prev_issue_date_per_grade", F.to_date(F.to_timestamp(F.col("prev_issue_date_per_grade"), "d MMMM yyyy")))
df.show()

+--------------------+--------------------+----------+--------------+----------+----------------+-------------------+--------+----------+-----------+-----------+-------+-----------+-----------+-----+-------------+----------+--------+-----+----------+----------+----------+------------------+--------------------+------------------+------------------+-------------------+-----------------------+------------------+--------------------+----------------------------+-----------------------------------+--------------------------------+--------------+---------------+---------------+----------+-----------+-------------+------------+-------------------------+
|         customer_id|           emp_title|emp_length|home_ownership|annual_inc|annual_inc_joint|verification_status|zip_code|addr_state|avg_cur_bal|tot_cur_bal|loan_id|loan_status|loan_amount|state|funded_amount|      term|int_rate|grade|issue_date|pymnt_plan|      type|           purpose|         description|emp_length_encoded|home_ownershi

### Previoius Loan amount from the same grade

In [32]:
df = df.withColumn('prev_loan_amount_per_grade', F.lag('loan_amount').over(window))
df.show()

+--------------------+--------------------+----------+--------------+----------+----------------+-------------------+--------+----------+-----------+-----------+-------+-----------+-----------+-----+-------------+----------+--------+-----+----------+----------+----------+------------------+--------------------+------------------+------------------+-------------------+-----------------------+------------------+--------------------+----------------------------+-----------------------------------+--------------------------------+--------------+---------------+---------------+----------+-----------+-------------+------------+-------------------------+--------------------------+
|         customer_id|           emp_title|emp_length|home_ownership|annual_inc|annual_inc_joint|verification_status|zip_code|addr_state|avg_cur_bal|tot_cur_bal|loan_id|loan_status|loan_amount|state|funded_amount|      term|int_rate|grade|issue_date|pymnt_plan|      type|           purpose|         description|emp_l

### Previous loan date from the same state and grade combined

In [33]:
window_state_grade = Window.partitionBy('state', 'grade').orderBy('issue_date')

In [34]:
df = df.withColumn('prev_issue_date_per_state_grade', F.lag('issue_date').over(window_state_grade))
df = df.withColumn("prev_issue_date_per_state_grade", F.to_date(F.to_timestamp(F.col("prev_issue_date_per_state_grade"), "d MMMM yyyy")))
df.show()

+--------------------+--------------------+----------+--------------+----------+----------------+-------------------+--------+----------+-----------+-----------+-------+-----------+-----------+-----+-------------+----------+--------+-----+----------+----------+----------+------------------+--------------------+------------------+------------------+-------------------+-----------------------+------------------+--------------------+----------------------------+-----------------------------------+--------------------------------+--------------+---------------+---------------+----------+-----------+-------------+------------+-------------------------+--------------------------+-------------------------------+
|         customer_id|           emp_title|emp_length|home_ownership|annual_inc|annual_inc_joint|verification_status|zip_code|addr_state|avg_cur_bal|tot_cur_bal|loan_id|loan_status|loan_amount|state|funded_amount|      term|int_rate|grade|issue_date|pymnt_plan|      type|           pu

### Previous loan amount from the same state and grade combined

In [35]:
df = df.withColumn('prev_loan_amount_per_state_grade', F.lag('loan_amount').over(window_state_grade))
df.show()

+--------------------+--------------------+----------+--------------+----------+----------------+-------------------+--------+----------+-----------+-----------+-------+-----------+-----------+-----+-------------+----------+--------+-----+----------+----------+----------+------------------+--------------------+------------------+------------------+-------------------+-----------------------+------------------+--------------------+----------------------------+-----------------------------------+--------------------------------+--------------+---------------+---------------+----------+-----------+-------------+------------+-------------------------+--------------------------+-------------------------------+--------------------------------+
|         customer_id|           emp_title|emp_length|home_ownership|annual_inc|annual_inc_joint|verification_status|zip_code|addr_state|avg_cur_bal|tot_cur_bal|loan_id|loan_status|loan_amount|state|funded_amount|      term|int_rate|grade|issue_date|py

# 5. Analysis SQL vs Spark

### 1. Identify the average loan amount and interest rate for loans marked as "Default" in the Loan Status, grouped by Emp Length and annual income ranges.

#### a) SQL

In [36]:
df.createOrReplaceTempView('fintech_data')

sql_query = '''
SELECT emp_length,
CASE
WHEN annual_inc < 20000 THEN 'Below 20k'
WHEN annual_inc >= 20000 AND annual_inc < 40000 THEN '20k-40k'
WHEN annual_inc >= 40000 AND annual_inc < 60000 THEN '40k-60k'
WHEN annual_inc >= 60000 AND annual_inc < 80000 THEN '60k-80k'
ELSE '80k+'
END AS income_range,
AVG(loan_amount) AS avg_loan_amount,
AVG(int_rate) AS avg_interest_rate
FROM fintech_data
WHERE loan_status = 'Default'
GROUP BY emp_length, income_range
ORDER BY emp_length, income_range;'''

avg_amount_interest_sql = spark.sql(sql_query)
avg_amount_interest_sql.show()

+----------+------------+---------------+-----------------+
|emp_length|income_range|avg_loan_amount|avg_interest_rate|
+----------+------------+---------------+-----------------+
|   5 years|     60k-80k|        25000.0|           0.1367|
+----------+------------+---------------+-----------------+



#### b) PySpark

In [37]:
default_loans = df.withColumn(
    "income_range",
    when(col("annual_inc") < 20_000, "Below 20k")
    .when((col("annual_inc") >= 20_000) & (col("annual_inc") < 40_000), "20k-40k")
    .when((col("annual_inc") >= 40_000) & (col("annual_inc") < 60_000), "40k-60k")
    .when((col("annual_inc") >= 60_000) & (col("annual_inc") < 80_000), "60k-80k")
    .otherwise("80k+")
)

default_loans = default_loans.filter(col('loan_status') == 'Default')

window_default_loans = Window.partitionBy('emp_length', 'income_range')

avg_amount_interest = default_loans\
    .withColumn('avg_loan_amount', avg('loan_amount').over(window_default_loans))\
    .withColumn('avg_int_rate', avg('int_rate').over(window_default_loans))

avg_amount_interest = avg_amount_interest.select('emp_length', 'income_range', 'avg_loan_amount', 'avg_int_rate').orderBy('emp_length', 'income_range')
avg_amount_interest.show()

+----------+------------+---------------+------------+
|emp_length|income_range|avg_loan_amount|avg_int_rate|
+----------+------------+---------------+------------+
|   5 years|     60k-80k|        25000.0|      0.1367|
+----------+------------+---------------+------------+



### 2. Calculate the average difference between Loan Amount and Funded Amount for each loan Grade and sort by the grades with the largest differences.

#### a) SQL

In [38]:
df.createOrReplaceTempView('fintech_data')

sql_query = '''
SELECT
letter_grade,
AVG(loan_amount - funded_amount) AS avg_difference
FROM fintech_data
GROUP BY letter_grade
ORDER BY avg_difference DESC;
'''

avg_amount_funded_diff_sql = spark.sql(sql_query)
avg_amount_funded_diff_sql.show()

+------------+--------------+
|letter_grade|avg_difference|
+------------+--------------+
|           F|           0.0|
|           E|           0.0|
|           B|           0.0|
|           D|           0.0|
|           C|           0.0|
|           A|           0.0|
|           G|           0.0|
+------------+--------------+



#### b) PySpark

In [39]:
window_diff = Window.partitionBy('letter_grade')

diff_df = df.withColumn('difference', df['loan_amount'] - df['funded_amount'])

diff_df = diff_df.withColumn('avg_difference', avg('difference').over(window_diff)).select("letter_grade", "avg_difference").distinct().orderBy(F.desc("avg_difference"))

diff_df.show()

+------------+--------------+
|letter_grade|avg_difference|
+------------+--------------+
|           A|           0.0|
|           B|           0.0|
|           C|           0.0|
|           D|           0.0|
|           E|           0.0|
|           F|           0.0|
|           G|           0.0|
+------------+--------------+



### 3. Compare the total Loan Amount for loans with "Verified" and "Not Verified" Verification Status across each state (Addr State).


#### a) SQL

In [40]:
df.createOrReplaceTempView('fintech_data')

sql_query = '''
SELECT
addr_state,
SUM(loan_amount) AS total_loan_amount
FROM fintech_data
WHERE verification_status IN ('Verified', 'Not Verified')
GROUP BY addr_state
ORDER BY total_loan_amount DESC
'''

total_amount_per_state_sql = spark.sql(sql_query)
total_amount_per_state_sql.show()

+----------+-----------------+
|addr_state|total_loan_amount|
+----------+-----------------+
|        CA|      3.4263475E7|
|        TX|       2.098395E7|
|        NY|        1.93881E7|
|        FL|       1.873515E7|
|        IL|      1.0755875E7|
|        NJ|        1.01946E7|
|        PA|        8735375.0|
|        VA|        8564175.0|
|        GA|        8111450.0|
|        OH|        7748325.0|
|        NC|        6614825.0|
|        MD|        6380450.0|
|        MI|        6258050.0|
|        AZ|        6098200.0|
|        MA|        5832450.0|
|        WA|        5184200.0|
|        CO|        5027275.0|
|        MN|        4356925.0|
|        NV|        4238525.0|
|        IN|        4206675.0|
+----------+-----------------+
only showing top 20 rows



#### b) PySpark

In [41]:
window_addr_state = Window.partitionBy('addr_state')

filtered_df = df.filter(df['verification_status'].isin('Verified', 'Not Verified'))

filtered_df = filtered_df.withColumn('total_loan_amount', F.sum('loan_amount').over(window_addr_state))

filtered_df = filtered_df.select('addr_state', 'total_loan_amount').distinct().orderBy('total_loan_amount', ascending=False)

filtered_df.show()

+----------+-----------------+
|addr_state|total_loan_amount|
+----------+-----------------+
|        CA|      3.4263475E7|
|        TX|       2.098395E7|
|        NY|        1.93881E7|
|        FL|       1.873515E7|
|        IL|      1.0755875E7|
|        NJ|        1.01946E7|
|        PA|        8735375.0|
|        VA|        8564175.0|
|        GA|        8111450.0|
|        OH|        7748325.0|
|        NC|        6614825.0|
|        MD|        6380450.0|
|        MI|        6258050.0|
|        AZ|        6098200.0|
|        MA|        5832450.0|
|        WA|        5184200.0|
|        CO|        5027275.0|
|        MN|        4356925.0|
|        NV|        4238525.0|
|        IN|        4206675.0|
+----------+-----------------+
only showing top 20 rows



### 4. Calculate the average time gap (in days) between consecutive loans for each grade using the new features you added in the feature engineering phase

#### a) SQL

In [42]:
df.createOrReplaceTempView('fintech_data')

sql_query = '''
SELECT
grade,
ROUND(AVG(DATEDIFF(issue_date, prev_issue_date_per_grade)), 2) AS avg_date_difference
FROM fintech_data
WHERE prev_issue_date_per_grade IS NOT NULL
GROUP BY grade
ORDER BY grade
'''

avg_time_gap_sql = spark.sql(sql_query)
avg_time_gap_sql.show()

+-----+-------------------+
|grade|avg_date_difference|
+-----+-------------------+
|    1|               2.34|
|    2|               2.22|
|    3|               2.34|
|    4|               2.32|
|    5|               2.28|
|    6|               1.63|
|    7|               1.69|
|    8|               1.69|
|    9|               1.62|
|   10|               1.69|
|   11|               1.76|
|   12|               1.82|
|   13|               1.74|
|   14|               1.68|
|   15|               1.74|
|   16|               3.56|
|   17|               3.58|
|   18|               3.45|
|   19|               3.49|
|   20|               3.37|
+-----+-------------------+
only showing top 20 rows



#### b) PySpark

In [43]:
avg_loan_date = df.withColumn('diff_date', F.date_diff(F.col('issue_date'), F.col('prev_issue_date_per_grade')))

avg_loan_date = avg_loan_date.filter(F.col('prev_issue_date_per_grade').isNotNull())

avg_loan_date = avg_loan_date.groupBy('grade').agg(F.round(F.avg('diff_date'), 2).alias('avg_time_gap_days'))

avg_loan_date.show()

+-----+-----------------+
|grade|avg_time_gap_days|
+-----+-----------------+
|    1|             2.34|
|    2|             2.22|
|    3|             2.34|
|    4|             2.32|
|    5|             2.28|
|    6|             1.63|
|    7|             1.69|
|    8|             1.69|
|    9|             1.62|
|   10|             1.69|
|   11|             1.76|
|   12|             1.82|
|   13|             1.74|
|   14|             1.68|
|   15|             1.74|
|   16|             3.56|
|   17|             3.58|
|   18|             3.45|
|   19|             3.49|
|   20|             3.37|
+-----+-----------------+
only showing top 20 rows



### 5. Identify the average difference in loan amounts between consecutive loans within the same state and grade combination.

#### a) SQL

In [44]:
df.createOrReplaceTempView('fintech_data')

sql_query = '''
SELECT
addr_state,
grade,
AVG(loan_amount - prev_loan_amount_per_state_grade) AS avg_loan_difference
FROM fintech_data
WHERE prev_loan_amount_per_state_grade IS NOT NULL
GROUP BY addr_state, grade
ORDER BY addr_state, grade
'''

avg_loan_diff_sql = spark.sql(sql_query)
avg_loan_diff_sql.show()

+----------+-----+-------------------+
|addr_state|grade|avg_loan_difference|
+----------+-----+-------------------+
|        AK|    1|            -2500.0|
|        AK|    3|             5000.0|
|        AK|    4|  4666.666666666667|
|        AK|    7|            -1500.0|
|        AK|    8| 3333.3333333333335|
|        AK|    9|            25000.0|
|        AK|   10|            18400.0|
|        AK|   11|-1779.1666666666667|
|        AK|   12|            -9000.0|
|        AK|   15|-1666.6666666666667|
|        AK|   16|             9625.0|
|        AK|   17|            -1550.0|
|        AK|   18|             7000.0|
|        AK|   19|           -11500.0|
|        AK|   20|             4187.5|
|        AK|   23|            -2400.0|
|        AL|    1|             -800.0|
|        AL|    2| 141.66666666666666|
|        AL|    3|            -1650.0|
|        AL|    4|  1142.857142857143|
+----------+-----+-------------------+
only showing top 20 rows



#### b) PySpark

In [45]:
avg_loan_diff = df.withColumn('loan_diff', F.col('loan_amount') - F.col('prev_loan_amount_per_state_grade'))

avg_loan_diff = avg_loan_diff.filter(F.col('prev_loan_amount_per_state_grade').isNotNull())

avg_loan_diff = avg_loan_diff.groupBy('addr_state', 'grade')\
    .agg(F.avg('loan_diff').alias('avg_loan_difference'))\
        .orderBy('addr_state', 'grade')

avg_loan_diff.show()

+----------+-----+-------------------+
|addr_state|grade|avg_loan_difference|
+----------+-----+-------------------+
|        AK|    1|            -2500.0|
|        AK|    3|             5000.0|
|        AK|    4|  4666.666666666667|
|        AK|    7|            -1500.0|
|        AK|    8| 3333.3333333333335|
|        AK|    9|            25000.0|
|        AK|   10|            18400.0|
|        AK|   11|-1779.1666666666667|
|        AK|   12|            -9000.0|
|        AK|   15|-1666.6666666666667|
|        AK|   16|             9625.0|
|        AK|   17|            -1550.0|
|        AK|   18|             7000.0|
|        AK|   19|           -11500.0|
|        AK|   20|             4187.5|
|        AK|   23|            -2400.0|
|        AL|    1|             -800.0|
|        AL|    2| 141.66666666666666|
|        AL|    3|            -1650.0|
|        AL|    4|  1142.857142857143|
+----------+-----+-------------------+
only showing top 20 rows



# 6. Lookup Table & Saving dataset

Already created in the `Encoding` Section

# 7. Saving Lookup Table & Dataset

In [46]:
df.coalesce(1).write.parquet("/data/fintech_spark_52_11870_clean.parquet")
lookup_df.coalesce(1).write.parquet("/data/lookup_spark_52_11870.parquet")